## TC 3007B
### GPT 2

<br>

#### Activity 2,3: Code GPT2
<br>

- Objective:
    - To understand the Transformer architecture.
    - To code GPT 2.
    - To gain understanding of the LLMs' autoregresive nature..
    
<br>

- Instructions

    This activity requires submission in teams. While teamwork is encouraged, each member is expected to contribute individually to the assignment. The final submission should feature the best arguments and solutions from each team member. Only one person per team needs to submit the completed work, but it is imperative that the names of all team members are listed in a Markdown cell at the very beginning of the notebook (either the first or second cell). Failure to include all team member names will result in the grade being awarded solely to the individual who submitted the assignment, with zero points given to other team members (no exceptions will be made to this rule).

    Follow the provided code. The code already implements a transformer from scratch as explained in [this video](https://youtu.be/51jq4wnHYaY)

    Since the provided code already implements a simple translator, your job for this assignment is to understand it fully, and document it using pictures, figures, and markdown cells.  
  
- Evaluation Criteria

    - Code Readability and Comments (40%).
    - Traning a LM,  complete 'Train function' (30%).
    - Generating at least 10 sentences, comple 'Sample function' (30%).

- Submission

Submit this Jupyter Notebook in canvas with your complete solution, ensuring your code is well-commented and includes Markdown cells that explain your design choices, results, and any challenges you encountered.




In [ ]:
!pip install transformers datasets

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import torch.optim as optim

In [2]:
class Config:
    '''
    Class Config with parameters for GPT2
    '''
    def __init__(self, vocab_size = 50257, max_seq_length = 128, embed_size = 768, num_layers = 12,
                 num_heads = 12, dropout = 0.1):
        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        self.embed_size = embed_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.dropout = dropout

In [7]:
class SelfAttention(nn.Module):
    '''
    
    
    '''
    
    def __init__(self, config):
        super().__init__()
        assert config.embed_size % config.num_heads == 0, 'sizes not compatible'
        self.num_heads = config.num_heads
        self.head_dim = config.embed_size // config.num_heads
        # linear transformation matrices
        self.W_q = nn.Linear(config.embed_size, config.embed_size)
        self.W_k = nn.Linear(config.embed_size, config.embed_size)
        self.W_v = nn.Linear(config.embed_size, config.embed_size)
        self.output = nn.Linear(config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)
        # lower triangular matrix
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(config.max_seq_length, config.max_seq_length)
                      ).view(-1, 1, config.max_seq_length, config.max_seq_length)
        )
    def forward(self, x):
        batch, seq_length, embed_dim = x.size() # B, T, D
        #
        Q = self.W_q(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2) #after trans B, numheads, T, head dim
        K = self.W_k(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        #
        attn = (Q@K.transpose(-2, -1))/(self.head_dim**0.5) #B, numheads, T, T
        attn = attn.masked_fill(self.mask[:, :, :seq_length, :seq_length] == 0, float('-inf'))
        attn = F.softmax(attn, dim = -1)
        attn = self.dropout(attn)
        scores = attn @ V #B, numheads, T, head_dim

        scores = scores.transpose(1, 2).contiguous().view(batch, seq_length, embed_dim) #B, T, embed_size

        return self.dropout(scores)
        
        

In [9]:
class FFN(nn.Module):
    '''
    
    '''
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.embed_size, 4 * config.embed_size)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(4 * config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc2(self.gelu(self.fc1(x)))
        return self.dropout(x)

In [10]:
class Transformer(nn.Module):
    '''
    
    '''
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.embed_size)
        self.attention = SelfAttention(config)
        self.norm2 = nn.LayerNorm(config.embed_size)
        self.mlp = FFN(config)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [12]:
class GPT2(nn.Module):
    '''
    
    '''
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_size)
        self.pos_embed = nn.Embedding(config.max_seq_length, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)
        self.transformers = nn.Sequential(*[Transformer(config) for _ in range(config.num_layers)])
        self.norm1 = nn.LayerNorm(config.embed_size)

    def forward(self, input_tokens):
        batch, seq_length = input_tokens.size()
        pos = torch.arange(0, seq_length, dtype = torch.long, device = input_tokens.device).unsqueeze(0)
        x = self.token_embed(input_tokens) + self.pos_embed(pos)
        x = self.dropout(x)
        x = self.transformers(x)
        x = self.norm1(x)

        return x @ self.token_embed.weight.t()
        
        

In [13]:
def train(model, loader, optimiser, epochs = 10):
    ### 2 Do !!!!!!

In [ ]:
def sample(model, device, tokenizer, prompt, length=50, temperature = 1.0):
    ### 2 Do !!!!!

# Example usage
print(sample(model, device, tokeniser, prompt="Un estudiante de doctorado", length=50))

In [ ]:
def sample(model, device, tokenizer, prompt, length=50, temperature = 1.0):
    model.eval()
    tokens = tokenizer.encode(prompt, return_tensors='pt').to(device)

    for _ in range(length):
        tokens_cond = tokens[:, -SEQ_LENTGH:]
        with torch.no_grad():
            logits = model(tokens_cond)
        next_token_logits = logits[:, -1, :] / temperature
        next_token = torch.multinomial(F.softmax(next_token_logits, dim = -1), num_samples = 1)
        tokens = torch.cat([tokens, next_token], dim = 1)
        
    print(tokens)
    return tokenizer.decode(tokens[0])

# Example usage
print(sample(model, device, tokeniser, prompt="Un estudiante de doctorado", length=50))

In [ ]:
for